![MOMENTUM · Hackatón de DRL · 02 Agentes](../img/banner-e02.jpg)

# **2. Agentes de RL**

En este cuaderno aprenderemos a entrenar agentes basados en **aprendizaje por refuerzo profundo** (DRL) utilizando el entorno `highway-env` y la librería `stable-baselines3`.

[Stable Baselines 3](https://stable-baselines3.readthedocs.io/en/master/) es una de las librerías de DRL más utilizadas, y contiene implementaciones de los principales algoritmos que componen el estado del arte. Estas implementaciones son totalmente compatibles con la interfaz de `gymnasium`, lo que facilita la configuración, entrenamiento y evaluación de diferentes tipos de agentes en entornos variados.

Comenzamos importando las librerías necesarias:

In [ ]:
import gymnasium as gym
import highway_env

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

def show_state(env):
    """
    Muestra el estado actual del entorno.
    """
    img = env.render()
    clear_output(wait=True)
    plt.imshow(img)
    plt.axis("off")
    plt.show()

## **2.1. Entrenamiento básico**

Vamos a definir un entorno que nos permita **entrenar** a nuestros agentes. Lo llamaremos `env`, y se corresponde con el entorno `highway-v0` que vimos en el cuaderno anterior:

In [ ]:
env = gym.make("highway-v0", render_mode="rgb_array")

Stable Baselines 3 ofrece diferentes algoritmos que podemos utilizar para abordar este problema de conducción autónoma. La clave es conocer los **espacios de acciones** que es capaz de manejar cada algoritmo, tal y como se detalla en su [documentación](https://stable-baselines3.readthedocs.io/en/master/guide/algos.html). De esta forma, sabremos si es posible utilizarlo en nuestro entorno.


| Algoritmo          | `Box` | `Discrete` | `MultiDiscrete` | `MultiBinary` |
| ------------------ | ----- | ---------- | --------------- | ------------- |
| `A2C`              | ✔️    | ✔️         | ✔️              | ✔️            |
| `DDPG`             | ✔️    | ❌         | ❌              | ❌            | 
| `DQN`              | ❌    | ✔️         | ❌              | ❌            |
| `HER`              | ✔️    | ✔️         | ❌              | ❌            | 
| `PPO`              | ✔️    | ✔️         | ✔️              | ✔️            |
| `SAC`              | ✔️    | ❌         | ❌              | ❌            | 
| `TD3`              | ✔️    | ❌         | ❌              | ❌            | 

Veamos qué significa cada **espacio de acciones**:

- `Box`: espacio continuo. Las acciones son números reales dentro de unos límites. Ejemplo: `[-1.0, 0.5, 0.2]`.
- `Discrete`: espacio discreto. El agente elige una única opción entre un número finito de posibilidades. Ejemplo: `Discrete(3) → {0, 1, 2}`.
- `MultiDiscrete`: varias variables discretas independientes, cada una con sus propias opciones. Ejemplo: `MultiDiscrete([3, 2])` → una acción como `[2, 1]`. Aquí la primera variable puede tomar los valores `0,1,2` y la segunda, `0,1`.
- `MultiBinary`: varias variables binarias, cada una solo puede ser `0` o `1`. Ejemplo: `MultiBinary(4) → [1, 0, 1, 1]`.

Si comprobamos el espacio de acciones de nuestro entorno, observamos que es del tipo `Discrete`, por lo que tenemos a nuestra disposición los siguientes algoritmos:

- `A2C`
- `DQN`
- `HER`
- `PPO`

También es posible "discretizar" algoritmos que trabajan con espacios tipo `Box`, pero de momento abordaremos el problema de la forma más sencilla posible.

Elegiremos, por ejemplo: [`A2C`](https://stable-baselines3.readthedocs.io/en/master/modules/a2c.html). Creamos nuestro agente de la siguiente manera:

In [ ]:
from stable_baselines3 import A2C

agent = A2C("MlpPolicy", env, verbose=1)

Puede verse como importamos `A2C` de la librería `stable_baselines3` y lo declaramos a continuación.

Todo agente requiere dos parámetros:

- El tipo de **política**. En este caso, `MlpPolicy` (_multi-layer perceptron_, o red neuronal).
- El **entorno** en el que se va a ejecutar. Esto permite al agente conocer el espacio de observaciones y acciones.

Una vez creado nuestro agente, procedemos a **entrenarlo**. Para ello, hacemos uso de la función `learn`, donde indicamos el número de pasos (_timesteps_) empleados para entrenar.

Entrenaremos el agente durante 40 timesteps. Se trata de un número de _timesteps_ muy reducido, por lo que no esperamos obtener un agente bien entrenado. Simplemente nos servirá para ilustrar el proceso de entrenamiento:

In [ ]:
TRAIN_TIMESTEPS = 40 # Numero de timesteps de entrenamiento
# Por defecto, la duracion maxima de un episodio es de 40 timesteps

agent.learn(TRAIN_TIMESTEPS)

Para **guardar** el modelo entrenado en un archivo, utilizamos `save`:

In [ ]:
agent.save("agentes/ejemplo_A2C")

También podemos **cargar** un modelo existente con `load`:

In [ ]:
agent = A2C.load("agentes/ejemplo_A2C") # Cargamos el modelo previamente guardado

Si queremos evaluar el modelo entrenado, la opción más sencilla es utlizar la función `evaluate_policy`, que recibe como argumentos el agente, entorno y número de episodios empleados para evaluar. El resultado es la **recompensa media** obtenida en cada episodio y la **desviación típica**:

In [ ]:
from stable_baselines3.common.evaluation import evaluate_policy

EVAL_EPISODES = 5 # Numero de episodios de evaluacion

mean_reward, std_reward = evaluate_policy(
    agent,
    env,
    n_eval_episodes=EVAL_EPISODES
)

print(f"Recompensa media: {mean_reward:.2f} ± {std_reward:.2f}")

También podemos evaluar a nuestro agente definiendo el bucle de interacción agente-entorno y calculando manualmente la recompensa media.

Para que el agente decida qué acciones ejecutar a partir del estado actual, utlizamos `predict`. Esta función, recibe como argumento el estado actual y devuelve la acción elegida por el agente. Si indicamos `deterministic=True`, cada estado se corresponderá siempre con la misma acción. Esto es especialmente necesario cuando evaluamos a un agente y no queremos que mantenga un comportamiento exploratorio / estocástico, como sí ocurre durante el entrenamiento.

In [ ]:
episode_rewards = []

for i in range(EVAL_EPISODES):
    state, _ = env.reset()

    terminated = truncated = False
    episode_reward = 0

    while not (terminated or truncated):

        # Utilizamos predict para obtener la accion a partir del estado actual
        action, _ = agent.predict(state, deterministic=True) # Predicciones deterministas

        state, reward, terminated, truncated, _ = env.step(action)

        episode_reward += reward

        show_state(env)

    episode_rewards.append(episode_reward)

env.close()

mean_reward = np.mean(episode_rewards)
std_reward = np.std(episode_rewards)

print(f"Recompensa media: {mean_reward:.2f} ± {std_reward:.2f}")

## **2.2. Hiperparámetros**

Todos los algoritmos de DRL pueden ser configurados especificando diferentes **_hiperparámetros_** que afectan al proceso de aprendizaje. Por ejemplo:

- El **factor de descuento**, que mide cuánto se valoran las recompensas futuras.
- Arquitectura de la **red neuronal** (número de capas ocultas, neuronas por capa, etc.).
- La **tasa de aprendizaje**, que indica cómo de "abruptas" son las actualizaciones de la red.
- **Tasa de exploración** (probabilidad de elegir una acción exploratoria).
- Tamaño de **_buffer_** / **_batch_**, que indica la cantidad de experiencias almacenadas en la memoria interna del agente.
- Etc.

Por defecto, para cada agente, Stable Baselines 3 ofrece una configuración de referencia que funciona de forma aceptable en una gran variedad de entornos. No obstante, estos hiperparámetros por defecto pueden no ser óptimos para un problema en específico. Es por esto que suele ser recomendable **calibrarlos** de forma manual, o empleando alguna librería de optimización como [Optuna](https://optuna.org/).

Veamos un ejemplo donde indicamos manualmente los valores de tasa de aprendizaje (`learning_rate`), factor de descuento (`gamma`) y tamaño de _buffer_ (`buffer_size`) para un agente **DQN**:

In [ ]:
from stable_baselines3 import DQN

agent = DQN("MlpPolicy", env, learning_rate=0.001, gamma=0.95, buffer_size=100000)

Para consultar los _hiperparámetros_ de un agente (así como otra información relevante), accedemos a su diccionario interno o, directamente, al parámetro que queremos visualizar:

In [ ]:
agent_config = agent.__dict__ # Diccionario interno con la configuracion del agente

# Acceso directo a los hiperparametros del agente
print("Learning rate: ", agent.learning_rate)
print("Buffer size: ", agent.buffer_size)
print("Gamma: ", agent.gamma)

## **2.3. Optimización automática de hiperparámetros**

A continuación, se muestra un ejemplo de **optimización automática** de hiperparámetros utilizando la librería **Optuna**.

- Comenzamos importando la librería `optuna`.
- Después, definimos una función `objective` que servirá para evaluar diferentes configuraciones de DQN. En el interior de la función, se definen diferentes combinaciones de valores para `learning_rate`, `gamma` y `buffer_size`.
- Creamos un _estudio_ indicando que buscamos maximiziar (`maximize`) la recompensa y ejecutamos `optimize` para probar 20 combinaciones.
- Tras la ejecución, obtenemos la mejor recompensa media obtenida y la combinación de hiperparámetros con la que se ha conseguido.

In [ ]:
import optuna

N_COMBINACIONES = 20

def objective(trial):

    # Hiperparametros que vamos a optimizar
    learning_rate = trial.suggest_float(
        "learning_rate", 1e-5, 1e-2, log=True
    )

    gamma = trial.suggest_float(
        "gamma", 0.90, 0.999
    )

    buffer_size = trial.suggest_categorical(
        "buffer_size", [10000, 50000, 100000]
    )

    # Crear agente
    agent = DQN(
        "MlpPolicy",
        env,
        learning_rate=learning_rate,
        gamma=gamma,
        buffer_size=buffer_size,
    )

    # Entrenamiento
    agent.learn(total_timesteps=TRAIN_TIMESTEPS)

    # Evaluacion
    mean_reward, std_reward = evaluate_policy(
        agent,
        env,
        n_eval_episodes=EVAL_EPISODES
    )

    return mean_reward


# Crear estudio de Optuna
study = optuna.create_study(direction="maximize")

# Probar 20 combinaciones
study.optimize(objective, n_trials=N_COMBINACIONES)

print("Mejores hiperparámetros:")
print(study.best_params)

print("Mejor recompensa:")
print(study.best_value)

## **2.4. _Callbacks_**

Finalmente, veamos qué son los _callbacks_ y cómo pueden ayudarnos en el entrenamiento y evaluación de los agentes.

Un _callback_ es una función que puede ser ejecutada a lo largo del proceso de entrenamiento. Algunas de sus utilidades son:

- Monitorizar el estado interno del agente durante el entrenamiento.
- Realizar _checkpoints_ (guardados automáticos) cada cierto número de pasos de tiempo.
- Realizar evaluaciones periódicas.
- Etc.

En la [documentación oficial de Stable Baselines 3](https://stable-baselines3.readthedocs.io/en/master/guide/callbacks.html) se muestra cómo crear un _callback_ y el listado de _callbacks_ preimplementados.

En nuestro caso, mostraremos dos ejemplos: [`EvalCallback`](https://stable-baselines3.readthedocs.io/en/master/guide/callbacks.html#evalcallback) y [`CheckpointCallback`](https://stable-baselines3.readthedocs.io/en/master/guide/callbacks.html#checkpointcallback):

- `EvalCallback` permite realizar evaluaciones periódicas del agente en un **entorno de evaluación aislado**. Esto permite mantener siempre la mejor versión del agente obtenida durante el entrenamiento.

- `CheckpointCallback` guarda el estado actual del modelo cada $N$ _timesteps_.

Si queremos emplear varios _callbacks_, los introducimos dentro de una **lista de _callbacks_** (`CallbackList`). 

Veamos un ejemplo:

In [ ]:
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback, CallbackList

EVAL_FREQ = 40  # Frecuencia de evaluacion
EVAL_EPISODES = 2 # Episodios de evaluacion
CHECKPOINT_FREQ = 40 # Frecuencia de guardado

# Definimos un entorno de evaluacion independiente al de entrenamiento
eval_env = gym.make("highway-v0", render_mode="rgb_array")

# Definimos el callback de evaluacion
eval_callback = EvalCallback(eval_env, best_model_save_path="./logs/",
                             log_path="./logs/", eval_freq=EVAL_FREQ,
                             n_eval_episodes=EVAL_EPISODES,
                             deterministic=True, render=False)

# Definimos el callback de guardado periodico
checkpoint_callback = CheckpointCallback(
  save_freq=CHECKPOINT_FREQ,
  save_path="./logs/",
  name_prefix="rl_agent",
)

# Definimos una lista de callbacks y los incluimos
callbacks = CallbackList([
    eval_callback,
    checkpoint_callback
])

Una vez definidos nuestros _callbacks_, los incorporamos al entrenamiento del agente y entrenamos con `learn`:

In [ ]:
agent = DQN("MlpPolicy", env)

agent.learn(
    total_timesteps=TRAIN_TIMESTEPS,
    callback=callbacks
)

El resultado será un directorio `logs` con información del entrenamiento, mejor modelo (`best_model.zip`) y guardados periódicos (`rl_agent_N_steps.zip`).

Finalmente, podemos cargar el mejor agente obtenido (`best_model.zip`) utilizando `load`. Esto permite utilizar la versión del agente que mejor rendimiento ha obtenido durante el entrenamiento, evitando utilizar una versión subóptima del mismo:

In [ ]:
best_agent = DQN.load("logs/best_model.zip")

A modo de conclusión, en este cuaderno hemos aprendido varios conceptos clave:

- Cómo se define un agente empleando `stable_baselines3`.
- Cómo configurar un agente y optimizar sus hiperparámetros con `optuna`.
- Cómo aplicar _callbacks_ durante el entrenamiento de nuestros agentes.

En el próximo cuaderno, estudiaremos otras herramientas avanzadas destinadas a mejorar el rendimiento de nuestro agentes.